# 06 - Evaluation: Statistik Komposisi & Proyeksi WMA## TujuanMenguji apakah komposisi topik berubah signifikan antar tahun, lalu memproyeksikantren distribusi topik 2026-2027 dengan Weighted Moving Average.## Asal (Input)- `_fix_final/topic_by_year_final.csv` (dari Notebook 08)- `_fix_final/topic_by_year_sensitivity.csv`- `_label_work/labels.json`## Keluaran (Output)- `_stats_forecast/forecast_table.csv` - proyeksi per topik- `_stats_forecast/backtest_results.csv` - MAE walk-forward- `_stats_forecast/share_intervals.csv` - interval Wilson 95%- `_stats_forecast/per_topic_tests.csv` - uji Spearman + Bonferroni- `_stats_forecast/composition_tests_report.md` - chi-square- `_stats_forecast/sensitivity_report.md` - uji sensitivitas## Metodologi ProyeksiWMA dengan bobot linear [1,2,3,4,5] diterapkan pada 5 titik historis (2021-2025).Proyeksi 2026 dihitung dari data asli; 2027 bersifat rekursif (indikatif).Karena hanya tersedia 5 titik, seluruh proyeksi **tidak dapat diklaimsebagai prediksi statistik yang andal**.

In [ ]:
import sys, io as _iofrom pathlib import Path_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()sys.path.insert(0, str(_ROOT))try:    sys.stdout.reconfigure(encoding='utf-8', errors='replace')except Exception:    passimport pandas as pd, numpy as npfrom src import config as CC.ensure_dirs()print(C.describe())

## Langkah 1 - Jalankan Analisis Statistik & Proyeksi

In [ ]:
from src import forecastforecast.main()

## Langkah 2 - Hasil Uji Komposisi (Chi-Square)

In [ ]:
print(open(str(C.COMPOSITION_REPORT), encoding='utf-8').read()[:800])

## Langkah 3 - Tabel Proyeksi WMA

In [ ]:
fc = pd.read_csv(C.FORECAST_CSV)cols = ['topik_id','label','share_2025','wma_2026','pita_bawah_2026',        'pita_atas_2026','wma_2027','arah']print(fc[cols].to_string(index=False))

## Langkah 4 - Visualisasi Proyeksi

In [ ]:
import plotly.graph_objects as gotby = pd.read_csv(C.TOPIC_BY_YEAR_FINAL)fig = go.Figure()for t in [3, 4, 5, 6]:    fig.add_trace(go.Scatter(        x=['2021','2022','2023','2024','2025'],        y=tby[f'proporsi_topik_{t}'],        mode='lines+markers', name=f'Topik {t}'))    row = fc[fc['topik_id']==t].iloc[0]    fig.add_trace(go.Scatter(        x=['2025','2026','2027'],        y=[tby[f'proporsi_topik_{t}'].iloc[-1], row['wma_2026'], row['wma_2027']],        mode='lines+markers', line=dict(dash='dot'),        name=f'Topik {t} proyeksi', showlegend=False))fig.add_vline(x=4, line_dash='dash', line_color='#999')fig.update_layout(template='plotly_white', height=460,                  xaxis_title='Tahun', yaxis_title='Proporsi (%)')fig.show()

## Langkah 5 - Hasil Walk-Forward Backtest

In [ ]:
bt = pd.read_csv(C.BACKTEST_CSV)wma = bt[bt['metode']=='WMA']hist= bt[bt['metode']=='Rata-rata Historis']print(f'MAE rerata WMA            : {wma["mae_pp"].mean():.2f} pp')print(f'MAE rerata Rata-rata Hist.: {hist["mae_pp"].mean():.2f} pp')print()better = (wma['mae_pp'].values < hist['mae_pp'].values).sum()print(f'WMA lebih akurat pada {better} dari 8 topik')print()print('Kesimpulan: WMA TIDAK secara konsisten mengalahkan baseline rerata,')print('sehingga seluruh proyeksi hanya bersifat indikatif.')

## Ringkasan Fase 06 - Evaluation & Forecast| Temuan | Nilai ||:---|---:|| Chi-square p-value | 0,6861 (tidak signifikan) || Cramér's V | 0,1448 (asosiasi lemah) || MAE rerata WMA | 4,64 pp || Topik dengan WMA lebih baik | 2 dari 8 |Tidak ditemukan pergeseran komposisi topik yang signifikan secara statistisselama 2021-2025. Proyeksi WMA hanya indikatif dan tidak boleh diklaimsebagai prediksi yang andal.

## Peta Artefak| Artefak | Dibuat oleh | Consumed by ||:---|:---|:---|| `_stats_forecast/forecast_table.csv` | Notebook 06 | app (halaman Proyeksi) || `_stats_forecast/backtest_results.csv` | Notebook 06 | Lampiran Bab IV || `_stats_forecast/share_intervals.csv` | Notebook 06 | app (halaman Tren) |